# Synthetic support-ticket generator

Week 3 exercise: build a labeled classification set with an open-source Hugging Face model running in 4-bit.

Each row is one customer support ticket for a single product, with two closed-set labels:

- `intent`: why the customer wrote in
- `urgency`: how soon it needs a response

You choose the label cell. The model only writes the ticket text. A validator keeps a row when the JSON parses, the labels match the cell you asked for, and the text is a plausible length. Everything else is rejected and retried.

**Run this notebook on Google Colab with a GPU.** 4-bit loading uses bitsandbytes, which needs an NVIDIA GPU. In Colab: Runtime → Change runtime type → T4 GPU.

Read each short note, then run the code cell under it. Later cells use names defined earlier, so run them from top to bottom. The first model download takes a few minutes. After that, generating the small default set (2 tickets × 18 label cells) is the long step.

## 1. Install the libraries

`transformers` loads the model and tokenizer. `accelerate` places the model on the GPU. `bitsandbytes` supplies the 4-bit quantization that `BitsAndBytesConfig` turns on.

Colab already has PyTorch. Re-running this cell is safe.

In [ ]:
%pip install -q transformers accelerate bitsandbytes

## 2. Imports and a GPU check

Stop here if CUDA is missing. Switch the runtime to a T4 and rerun from the top. The quantization config in the next steps follows the Hugging Face Transformers and bitsandbytes docs: NF4 4-bit weights, float16 compute, and `device_map="auto"`.

In [ ]:
import json
import random
from collections import Counter

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. In Colab, set Runtime → Change runtime type → T4 GPU, then rerun from the top.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("PyTorch:", torch.__version__)

## 3. Product, labels, and generation settings

Edit `PRODUCT` if you want tickets for something else. Keep one product for the whole run so the wording stays consistent.

The label lists are closed. A ticket is stored only when `intent` and `urgency` are exactly the pair requested for that cell.

`TICKETS_PER_CELL` is 2 so you can finish a first pass and read the results. There are 6 intents × 3 urgency levels = 18 cells, so the default run stores about 36 tickets. Set it to 10 when you want the full balanced set of 180.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
PRODUCT = "Northstar, a project-management web app for small teams"

INTENTS = [
    "billing",
    "account_access",
    "bug_report",
    "how_to",
    "cancellation",
    "feature_request",
]
URGENCIES = ["low", "medium", "high"]

URGENCY_MEANINGS = {
    "low": "a question or minor inconvenience, with no deadline",
    "medium": "something is broken or blocking, and a workaround exists",
    "high": "a money, access, or data problem that needs action the same day",
}

DETAIL_HINTS = [
    "a plan name such as Free, Pro, or Business",
    "a time of day or a day of the week",
    "a short error message the customer saw on screen",
    "a project or workspace name",
    "a teammate's role, such as admin or guest",
    "an amount of money or a seat count",
]

TICKETS_PER_CELL = 2
MAX_RETRIES = 3
MAX_NEW_TOKENS = 180
TEMPERATURE = 0.8
OUTPUT_PATH = "support_tickets.jsonl"

random.seed(42)
torch.manual_seed(42)

print(f"Model: {MODEL_ID}")
print(f"Cells: {len(INTENTS) * len(URGENCIES)}  |  tickets per cell: {TICKETS_PER_CELL}")
print(f"Target rows: {len(INTENTS) * len(URGENCIES) * TICKETS_PER_CELL}")

## 4. Load the tokenizer and the 4-bit model

NF4 stores the weights in 4 bits. `bnb_4bit_compute_dtype=torch.float16` is the dtype used while the model runs; the stored weights stay 4-bit. On a T4 that brings a 3B instruct model down to a few gigabytes.

`Qwen/Qwen2.5-3B-Instruct` is public, so this cell does not need a Hugging Face token. The download happens on the first run.

To try Llama instead, set `MODEL_ID` to `meta-llama/Llama-3.2-3B-Instruct`, accept the license on the model page, and log in with `huggingface_hub.login()` before this cell.

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
)
model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
print(f"Loaded {MODEL_ID}")
print(f"Parameter memory: {param_bytes / 1024**3:.2f} GB")
print(f"Device map: {getattr(model, 'hf_device_map', model.device)}")

## 5. Build the prompt for one label cell

The chat template turns a system message and a user message into the exact string this model was trained to continue. `add_generation_prompt=True` leaves the reply open for the assistant.

The user message fills in the intent, the urgency, and a detail hint. Asking for one ticket in one cell keeps a small quantized model on schema. The detail hint changes every call so the tickets do not all sound the same.

In [ ]:
SYSTEM_PROMPT = (
    "You write realistic customer support tickets. "
    "Return only a JSON object with the keys text, intent, and urgency. "
    "Do not add markdown fences or extra keys."
)


def build_messages(intent: str, urgency: str, detail_hint: str) -> list[dict]:
    user = f"""Product: {PRODUCT}
Write one customer support ticket.
intent must be exactly: {intent}
urgency must be exactly: {urgency}

Urgency meanings:
- low: {URGENCY_MEANINGS["low"]}
- medium: {URGENCY_MEANINGS["medium"]}
- high: {URGENCY_MEANINGS["high"]}

The ticket is 1-3 sentences, written by the customer, in the first person.
Include this kind of detail: {detail_hint}
The wording of the ticket must match the urgency above.
Do not use the label names inside the ticket text.
Return only JSON."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]


sample_messages = build_messages("bug_report", "high", DETAIL_HINTS[0])
sample_prompt = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=True,
)
print(sample_prompt)

## 6. Generate one raw completion

This is the model call by itself. The tokenizer encodes the chat prompt, `model.generate` samples a continuation, and we decode only the new tokens so the prompt is not part of the text we parse.

Temperature 0.8 keeps the tickets varied. Run this cell more than once and the wording should change. The next section decides whether this particular completion is good enough to keep.

In [ ]:
def generate_raw(intent: str, urgency: str, detail_hint: str) -> str:
    messages = build_messages(intent, urgency, detail_hint)
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_token_ids = output_ids[0, model_inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(new_token_ids, skip_special_tokens=True)


raw_ticket = generate_raw("bug_report", "high", DETAIL_HINTS[2])
print(raw_ticket)

## 7. Parse the JSON and apply the closed-set checks

Small quantized models sometimes wrap JSON in prose, invent an extra label, or repeat a ticket. These checks turn that into a reject reason you can read:

- the completion contains one JSON object with `text`, `intent`, and `urgency`
- `intent` and `urgency` match the cell you requested
- `text` is between 20 and 400 characters
- the label names do not appear inside the ticket text
- the same text has not already been stored

Run this cell right after step 6. If the reason is `None`, that completion would be kept.

In [ ]:
LABEL_WORDS = set(INTENTS) | {"urgency", "intent"}


def extract_json(text: str):
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end <= start:
        return None
    try:
        return json.loads(text[start : end + 1])
    except json.JSONDecodeError:
        return None


def reject_reason(row, intent: str, urgency: str, seen: set[str]):
    if not isinstance(row, dict):
        return "not a JSON object"
    if not {"text", "intent", "urgency"}.issubset(row):
        return "missing text, intent, or urgency"
    ticket_text = row["text"]
    if not isinstance(ticket_text, str):
        return "text is not a string"
    ticket_text = " ".join(ticket_text.split())
    if not 20 <= len(ticket_text) <= 400:
        return f"text length {len(ticket_text)} is outside 20-400"
    if row["intent"] != intent or row["urgency"] != urgency:
        return f"labels {row['intent']!r} / {row['urgency']!r} do not match {intent} / {urgency}"
    lowered = ticket_text.lower()
    leaked = [word for word in LABEL_WORDS if word in lowered]
    if leaked:
        return f"label name leaked into text: {leaked[0]}"
    if lowered in seen:
        return "duplicate text"
    return None


parsed = extract_json(raw_ticket)
print("Parsed:", json.dumps(parsed, indent=2) if parsed else None)
print("Reject reason:", reject_reason(parsed, "bug_report", "high", set()))

## 8. Retry until one ticket passes

`make_ticket` generates, parses, and validates. It retries up to `MAX_RETRIES` times. Each attempt is printed, including the reject reason, so you can see what the quantized model got wrong.

The stored record uses our labels and a cleaned `text`. The ticket id is assigned by this notebook, not by the model.

In [ ]:
def make_ticket(intent: str, urgency: str, detail_hint: str, seen: set[str], ticket_id: str):
    attempts = []
    for attempt in range(1, MAX_RETRIES + 1):
        raw = generate_raw(intent, urgency, detail_hint)
        row = extract_json(raw)
        reason = reject_reason(row, intent, urgency, seen)
        attempts.append(reason or "kept")
        print(f"  attempt {attempt}: {attempts[-1]}")
        if reason is None:
            text = " ".join(row["text"].split())
            seen.add(text.lower())
            return {
                "ticket_id": ticket_id,
                "text": text,
                "intent": intent,
                "urgency": urgency,
            }, attempts
    return None, attempts


seen_texts = set()
example, example_attempts = make_ticket(
    "billing",
    "high",
    DETAIL_HINTS[5],
    seen_texts,
    "T0000",
)
print(json.dumps(example, indent=2))

## 9. Generate a balanced set

The loop walks every intent and every urgency, and asks for `TICKETS_PER_CELL` keepers in that cell. The detail hint rotates through the hint list.

If a cell cannot produce a valid ticket within the retries, that slot is skipped and counted. You will fill gaps by rerunning this cell later, or by raising `MAX_RETRIES`. With the default of 2 per cell this takes several minutes on a T4.

In [ ]:
records = []
seen_texts = set()
reject_counts = Counter()
hint_index = 0

for intent in INTENTS:
    for urgency in URGENCIES:
        print(f"\n{intent} / {urgency}")
        kept = 0
        while kept < TICKETS_PER_CELL:
            hint = DETAIL_HINTS[hint_index % len(DETAIL_HINTS)]
            hint_index += 1
            ticket_id = f"T{len(records) + 1:04d}"
            ticket, attempts = make_ticket(intent, urgency, hint, seen_texts, ticket_id)
            for reason in attempts:
                if reason != "kept":
                    reject_counts[reason] += 1
            if ticket is None:
                print("  skipped this slot after retries")
                break
            records.append(ticket)
            kept += 1

print(f"\nKept {len(records)} tickets")
print("Reject reasons:")
for reason, count in reject_counts.most_common():
    print(f"  {count:3d}  {reason}")

## 10. Save JSONL and print the label counts

JSONL is one JSON object per line. That is the file you can load later for a classifier or a manual review.

The table should be almost even. A short cell means that pair was hard for the model and several retries were rejected. In Colab, download `support_tickets.jsonl` from the file browser when you want a copy outside the session.

In [ ]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as handle:
    for record in records:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Wrote {len(records)} rows to {OUTPUT_PATH}\n")
header = "intent".ljust(18) + "".join(urgency.rjust(8) for urgency in URGENCIES)
print(header)
print("-" * len(header))
for intent in INTENTS:
    counts = [
        sum(1 for record in records if record["intent"] == intent and record["urgency"] == urgency)
        for urgency in URGENCIES
    ]
    print(intent.ljust(18) + "".join(f"{count:8d}" for count in counts))

## 11. Read a sample of each cell

Check the wording, not only the JSON. A `high` ticket should describe a same-day money, access, or data problem. A `low` ticket should be a question or a minor inconvenience. If a whole cell sounds wrong, tighten `URGENCY_MEANINGS` and regenerate that pair.

When this looks right, raise `TICKETS_PER_CELL` to 10 and rerun from step 9. The same loader, quant config, and validator are what you would reuse for a Q&A generator later: change the prompt and the required JSON keys, and keep the reject-and-retry loop.

In [ ]:
for intent in INTENTS:
    for urgency in URGENCIES:
        match = next(
            (record for record in records if record["intent"] == intent and record["urgency"] == urgency),
            None,
        )
        print(f"[{intent} / {urgency}]")
        print(match["text"] if match else "(no ticket kept)")
        print()